In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ukb_utils import samples
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh37')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa037.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
in_dir_phased = "/well/lindgren/UKBIOBANK/nbaya/wes_200k/phase_ukb_wes/data/phased/non_singleton"

input_unphased_path = 'data/unphased/post-qc/ukb_wes_200k_filtered_chr21.mt'
input_phased_path = in_dir_phased + '/ukb_wes_phased_non_singleton_chr21-24xlong.qc-v4.2.2.vcf.gz'

In [4]:
# get tables
mt1 = qc.get_table(input_path=input_phased_path, input_type='vcf') # 12788
mt2 = qc.get_table(input_path=input_unphased_path, input_type='mt') # 11867 (for singletons

In [6]:

# Remove withdrawn samples
mt1 = samples.remove_withdrawn(mt1)
mt2 = samples.remove_withdrawn(mt2)

# filter by Duncan's final samples
#if final_sample_list:
#    ht_final_samples = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter=',')
#    mt1 = mt1.filter_cols(hl.is_defined(ht_final_samples[mt1.col_key]))
#    mt2 = mt2.filter_cols(hl.is_defined(ht_final_samples[mt2.col_key]))

# filter by Ducan's final variants
#if final_variant_list:
#    ht_final_variants = hl.import_table(final_variant_list, types={'locus':hl.tlocus(reference_genome='GRCh38'), 'alleles':hl.tarray(hl.tstr)})
#    ht_final_variants = ht_final_variants.key_by(ht_final_variants.locus, ht_final_variants.alleles)
#    mt1 = mt1.filter_rows(hl.is_defined(ht_final_variants[mt1.row_key]))
#    mt2 = mt2.filter_rows(hl.is_defined(ht_final_variants[mt2.row_key]))

# remove singletons since these can't be phased.
mt1 = qc.filter_min_mac(mt1, 2)
mt2 = qc.filter_min_mac(mt2, 2)
    
# load pedigree
fam_path = samples.get_fam_path(app_id=11867,wes_200k_only=False,relateds_only=False) # Files with True does not exist
pedigree = hl.Pedigree.read(fam_path)

# setup trip matriax
trio_dataset = hl.trio_matrix(mt2, pedigree, complete_trios=True)
mt = hl.experimental.phase_trio_matrix_by_transmission(trio_dataset)
mt = hl.experimental.explode_trio_matrix(mt)
    
# get call stats
#info_field = 'info'
#mt = mt.annotate_rows(**{info_field: hl.agg.call_stats(mt.PBT_GT, mt.alleles)})

# annotate shapeit4 GTs with GTs phased by transmission and filter to hets
#mt = mt.annotate_entries(GT_shapeit4 = mt1[mt.row_key, mt.col_key].GT)
#mt = mt.filter_entries(mt.GT_shapeit4.is_het_ref())
#mt = mt.filter_entries(mt.PBT_GT.is_het_ref())

# annotate switch errors
#mt = mt.annotate_rows(errors_sum=hl.agg.sum(mt.PBT_GT != mt.GT_shapeit4))
#mt = mt.annotate_rows(hets_sum=hl.agg.sum(mt.PBT_GT.is_het_ref()))
#mt = mt.annotate_rows(switch_errors=(mt.errors_sum / mt.hets_sum))

2021-12-07 21:15:17 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (user-supplied)
2021-12-07 21:15:17 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (user-supplied)
2021-12-07 21:16:20 Hail: WARN: Found 1 samples with missing sex information (not 1 or 2).
 Missing samples: [{'IID'}]


In [18]:
m = mt
m = m.drop('source_trio')
m = m.select_entries('PBT_GT')
m = m.annotate_entries(GT = m.PBT_GT)
m = m.select_entries('GT')
m = m.drop('info')
m.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
----------------------------------------
Entry fields:
    'GT': call
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [62]:
test = mt.select_entries(mt.PBT_GT, mt.GT, mt.GT_shapeit4)
test = test.drop(test.qual, test.filters)
test.entries().flatten().describe()#export(out_prefix + '_big.txt.gz')
 

----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'rsid': str 
    'info.AC': array<int32> 
    'info.AF': array<float64> 
    'info.AN': int32 
    'info.homozygote_count': array<int32> 
    'errors_sum': int64 
    'hets_sum': int64 
    'switch_errors': float64 
    'source_trio.id': str 
    'source_trio.proband.s': str 
    'source_trio.father.s': str 
    'source_trio.mother.s': str 
    'source_trio.is_female': bool 
    'source_trio.fam_id': str 
    's': str 
    'PBT_GT': call 
    'GT': call 
    'GT_shapeit4': call 
----------------------------------------
Key: []
----------------------------------------


----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'rsid': str 
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    } 
    'errors_sum': int64 
    'hets_sum': int64 
    'switch_errors': float64 
    'source_trio': struct {
        id: str, 
        proband: struct {
            s: str
        }, 
        father: struct {
            s: str
        }, 
        mother: struct {
            s: str
        }, 
        is_female: bool, 
        fam_id: str
    } 
    's': str 
    'PBT_GT': call 
    'GT': call 
    'GT_shapeit4': call 
----------------------------------------
Key: ['locus', 'alleles', 's']
----------------------------------------


In [5]:
# Get imputed data instead

In [7]:
mts = list()



for chrom in [21]:
    ht = genotypes.get_ukb_imputed_v3_mfi(chrom)
    ht = ht.annotate(ref = hl.if_else(ht.f6 == ht.a1, ht.a2, ht.a1))
    ht = ht.annotate(variant = hl.delimit([hl.str(chrom), hl.str(ht.position), ht.ref, ht.f6], ':'))
    ht = ht.key_by(**hl.parse_variant(ht.variant,  reference_genome= 'GRCh37'))
    mt = genotypes.get_ukb_imputed_v3_bgen([chrom])
    mt = mt.annotate_rows(imp_info = ht[mt.row_key].info)
    mts.append(mt)
    
mt = mts[0].union_rows(*mts[1:])
mt = mt.filter_rows(mt.imp_info > 0.8)


2021-12-08 11:04:37 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)
2021-12-08 11:04:38 Hail: INFO: Number of BGEN files parsed: 1
2021-12-08 11:04:38 Hail: INFO: Number of samples in BGEN files: 487409
2021-12-08 11:04:38 Hail: INFO: Number of variants across all BGEN files: 1261158


,,,,,,,,,,,
varid,rsid,position,a1,a2,maf,f6,info,ref,variant,locus,alleles
str,str,int32,str,str,float64,str,float64,str,str,locus<GRCh37>,array<str>
"""21:9411239_G_A""","""rs559462325""",9411239,"""G""","""A""",3.07e-05,"""A""",2.61e-01,"""G""","""21:9411239:G:A""",21:9411239,"[""G"",""A""]"
"""21:9411245_C_A""","""rs181691356""",9411245,"""C""","""A""",2.40e-03,"""A""",3.96e-01,"""C""","""21:9411245:C:A""",21:9411245,"[""C"",""A""]"
"""21:9411264_A_C""","""rs548263598""",9411264,"""A""","""C""",2.30e-06,"""C""",4.46e-01,"""A""","""21:9411264:A:C""",21:9411264,"[""A"",""C""]"
"""21:9411267_G_T""","""rs561987868""",9411267,"""G""","""T""",7.44e-05,"""T""",3.95e-02,"""G""","""21:9411267:G:T""",21:9411267,"[""G"",""T""]"
"""21:9411298_G_A""","""21:9411298_G_A""",9411298,"""G""","""A""",8.07e-05,"""A""",7.98e-02,"""G""","""21:9411298:G:A""",21:9411298,"[""G"",""A""]"
"""21:9411302_G_T""","""rs531010746""",9411302,"""G""","""T""",2.72e-04,"""T""",1.17e-01,"""G""","""21:9411302:G:T""",21:9411302,"[""G"",""T""]"
"""21:9411313_G_A""","""rs550852792""",9411313,"""G""","""A""",4.20e-05,"""A""",2.21e-01,"""G""","""21:9411313:G:A""",21:9411313,"[""G"",""A""]"
"""21:9411332_G_T""","""rs571137411""",9411332,"""G""","""T""",2.37e-04,"""T""",1.34e-01,"""G""","""21:9411332:G:T""",21:9411332,"[""G"",""T""]"


In [8]:
mt.count()

(1261158, 487409)

In [12]:
mt.aggregate_rows(hl.agg.sum(hl.is_defined(mt.imp_info)))

2021-12-08 11:05:45 Hail: INFO: Ordering unsorted dataset with network shuffle


1226472

In [40]:
chromosomes = ['20','21']

hts = list()
for chrom in chromosomes:
    ht = genotypes.get_ukb_imputed_v3_mfi(chrom)
    ht = ht.annotate(chrom = chrom)
    hts.append(ht)
    
ht = hts[0].union(*hts[1:])
ht = ht.annotate(ref = hl.if_else(ht.f6 == ht.a1, ht.a2, ht.a1))
ht = ht.annotate(variant = hl.delimit([hl.str(chrom), hl.str(ht.position), ht.ref, ht.f6], ':'))
ht = ht.key_by(**hl.parse_variant(ht.variant,  reference_genome= 'GRCh37'))
ht = ht.filter(ht.info >= 0.8)

2021-12-08 11:47:58 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)
2021-12-08 11:47:58 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)


In [39]:
mt = genotypes.get_ukb_imputed_v3_bgen(chromosomes)
mt = mt.filter_rows(hl.is_defined(ht[mt.row_key]))


2021-12-08 11:35:49 Hail: INFO: Number of BGEN files parsed: 2
2021-12-08 11:35:49 Hail: INFO: Number of samples in BGEN files: 487409
2021-12-08 11:35:49 Hail: INFO: Number of variants across all BGEN files: 3343729


In [14]:
ht.show()

2021-12-08 11:25:37 Hail: INFO: Ordering unsorted dataset with network shuffle
2021-12-08 11:25:40 Hail: INFO: Ordering unsorted dataset with network shuffle


,,,,,,,
varid,rsid,position,a1,a2,maf,f6,info
str,str,int32,str,str,float64,str,float64
"""20:10000054_CTTTG_C""","""20:10000054_CTTTG_C""",10000054,"""CTTTG""","""C""",1.88e-04,"""C""",3.67e-01
"""20:10000419_C_T""","""20:10000419_C_T""",10000419,"""C""","""T""",3.89e-05,"""T""",3.47e-01
"""20:10001373_C_T""","""20:10001373_C_T""",10001373,"""C""","""T""",1.80e-05,"""T""",4.37e-02
"""20:10001550_C_T""","""20:10001550_C_T""",10001550,"""C""","""T""",1.23e-04,"""T""",3.54e-01
"""20:10001958_A_G""","""20:10001958_A_G""",10001958,"""A""","""G""",1.85e-05,"""G""",6.63e-01
"""20:10002289_T_C""","""20:10002289_T_C""",10002289,"""T""","""C""",4.72e-05,"""C""",1.55e-01
"""20:10002355_CTT_C""","""20:10002355_CTT_C""",10002355,"""CTT""","""C""",2.27e-04,"""C""",5.26e-01
"""20:10002537_T_C""","""20:10002537_T_C""",10002537,"""T""","""C""",1.33e-04,"""C""",6.33e-01


2021-12-08 11:26:40 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)


In [29]:
ht = genotypes.get_ukb_imputed_v3_mfi(chrom)
ht = ht.annotate(varid_to_variant = ht.varid.replace('_',':'))
ht = ht.annotate(ref = hl.if_else(ht.f6 == ht.a1, ht.a2, ht.a1))
ht = ht.annotate(variant = hl.delimit([hl.str(chrom), hl.str(ht.position), ht.ref, ht.f6], ':'))
ht = ht.key_by(**hl.parse_variant(ht.variant,  reference_genome= 'GRCh37'))


#ht = ht.annotate(variant = hl.delimit([hl.str(chrom), hl.str(ht.position), ht.ref, ht.f6], ':'))


2021-12-08 11:29:47 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)


In [30]:
ht.show()

,,,,,,,,,,,,
varid,rsid,position,a1,a2,maf,f6,info,varid_to_variant,ref,variant,locus,alleles
str,str,int32,str,str,float64,str,float64,str,str,str,locus<GRCh37>,array<str>
"""21:9411239_G_A""","""rs559462325""",9411239,"""G""","""A""",3.07e-05,"""A""",2.61e-01,"""21:9411239:G:A""","""G""","""21:9411239:G:A""",21:9411239,"[""G"",""A""]"
"""21:9411245_C_A""","""rs181691356""",9411245,"""C""","""A""",2.40e-03,"""A""",3.96e-01,"""21:9411245:C:A""","""C""","""21:9411245:C:A""",21:9411245,"[""C"",""A""]"
"""21:9411264_A_C""","""rs548263598""",9411264,"""A""","""C""",2.30e-06,"""C""",4.46e-01,"""21:9411264:A:C""","""A""","""21:9411264:A:C""",21:9411264,"[""A"",""C""]"
"""21:9411267_G_T""","""rs561987868""",9411267,"""G""","""T""",7.44e-05,"""T""",3.95e-02,"""21:9411267:G:T""","""G""","""21:9411267:G:T""",21:9411267,"[""G"",""T""]"
"""21:9411298_G_A""","""21:9411298_G_A""",9411298,"""G""","""A""",8.07e-05,"""A""",7.98e-02,"""21:9411298:G:A""","""G""","""21:9411298:G:A""",21:9411298,"[""G"",""A""]"
"""21:9411302_G_T""","""rs531010746""",9411302,"""G""","""T""",2.72e-04,"""T""",1.17e-01,"""21:9411302:G:T""","""G""","""21:9411302:G:T""",21:9411302,"[""G"",""T""]"
"""21:9411313_G_A""","""rs550852792""",9411313,"""G""","""A""",4.20e-05,"""A""",2.21e-01,"""21:9411313:G:A""","""G""","""21:9411313:G:A""",21:9411313,"[""G"",""A""]"
"""21:9411332_G_T""","""rs571137411""",9411332,"""G""","""T""",2.37e-04,"""T""",1.34e-01,"""21:9411332:G:T""","""G""","""21:9411332:G:T""",21:9411332,"[""G"",""T""]"


In [ ]:
ht = genotypes.get_ukb_imputed_v3_mfi(chrom)